# Build GPT — full working reference notebook

This is a complete, runnable GPT-2-like transformer on Tiny Shakespeare. Type it along with Karpathy's video — do not just run cells. Your fingers need the reps.

**What's here:**
- Tokenizer, data loader, train/val split
- Self-attention (single head, then multi-head)
- Transformer block with residuals + pre-norm LayerNorm
- Training loop with loss curve plot
- **Attention pattern visualization** — see what the model learned to attend to

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

torch.manual_seed(1337)
device = 'cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu'
print('device:', device)

In [ ]:
# Hyperparameters (start small; scale up when it works)
block_size   = 128
batch_size   = 32
n_embd       = 192
n_head       = 6
n_layer      = 4
dropout      = 0.1
learning_rate= 3e-4
max_iters    = 3000
eval_interval= 300
eval_iters   = 50
assert n_embd % n_head == 0

In [ ]:
# Data — Tiny Shakespeare
# !wget -q https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt
text = open('input.txt').read()
chars = sorted(list(set(text)))
vocab_size = len(chars)
stoi = {c: i for i, c in enumerate(chars)}
itos = {i: c for c, i in stoi.items()}
encode = lambda s: [stoi[c] for c in s]
decode = lambda l: ''.join(itos[i] for i in l)
print(f'chars: {len(text)} | vocab: {vocab_size}')

data = torch.tensor(encode(text), dtype=torch.long)
n = int(0.9 * len(data))
train_data, val_data = data[:n], data[n:]

def get_batch(split):
    d = train_data if split == 'train' else val_data
    ix = torch.randint(len(d) - block_size, (batch_size,))
    x = torch.stack([d[i:i + block_size] for i in ix])
    y = torch.stack([d[i + 1:i + 1 + block_size] for i in ix])
    return x.to(device), y.to(device)

## The transformer, from parts

In [ ]:
class Head(nn.Module):
    """One head of self-attention."""
    def __init__(self, head_size):
        super().__init__()
        self.key   = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))
        self.dropout = nn.Dropout(dropout)
        self._last_attn = None  # stash for visualization

    def forward(self, x):
        B, T, C = x.shape
        k = self.key(x)      # (B, T, hs)
        q = self.query(x)    # (B, T, hs)
        wei = q @ k.transpose(-2, -1) * k.shape[-1] ** -0.5   # (B, T, T)
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf'))
        wei = F.softmax(wei, dim=-1)
        self._last_attn = wei.detach()
        wei = self.dropout(wei)
        v = self.value(x)
        return wei @ v

class MultiHeadAttention(nn.Module):
    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        self.proj = nn.Linear(head_size * num_heads, n_embd)
        self.dropout = nn.Dropout(dropout)
    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        return self.dropout(self.proj(out))

class FeedForward(nn.Module):
    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.ReLU(),
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(dropout),
        )
    def forward(self, x): return self.net(x)

class Block(nn.Module):
    def __init__(self, n_embd, n_head):
        super().__init__()
        head_size = n_embd // n_head
        self.sa   = MultiHeadAttention(n_head, head_size)
        self.ffwd = FeedForward(n_embd)
        self.ln1  = nn.LayerNorm(n_embd)
        self.ln2  = nn.LayerNorm(n_embd)
    def forward(self, x):
        x = x + self.sa(self.ln1(x))      # pre-norm + residual
        x = x + self.ffwd(self.ln2(x))
        return x

class GPT(nn.Module):
    def __init__(self):
        super().__init__()
        self.token_embedding    = nn.Embedding(vocab_size, n_embd)
        self.position_embedding = nn.Embedding(block_size, n_embd)
        self.blocks = nn.Sequential(*[Block(n_embd, n_head) for _ in range(n_layer)])
        self.ln_f   = nn.LayerNorm(n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        tok = self.token_embedding(idx)                             # (B, T, C)
        pos = self.position_embedding(torch.arange(T, device=idx.device))  # (T, C)
        x = tok + pos
        x = self.blocks(x)
        x = self.ln_f(x)
        logits = self.lm_head(x)
        if targets is None:
            return logits, None
        loss = F.cross_entropy(logits.view(-1, vocab_size), targets.view(-1))
        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -block_size:]
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :]
            probs = F.softmax(logits, dim=-1)
            next_idx = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, next_idx), dim=1)
        return idx

model = GPT().to(device)
n_params = sum(p.numel() for p in model.parameters())
print(f'parameters: {n_params/1e6:.2f}M')

In [ ]:
@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            _, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean().item()
    model.train()
    return out

optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)
train_losses, val_losses, xs = [], [], []

for step in range(max_iters):
    if step % eval_interval == 0 or step == max_iters - 1:
        losses = estimate_loss()
        train_losses.append(losses['train']); val_losses.append(losses['val']); xs.append(step)
        print(f"step {step:5d} | train {losses['train']:.4f} | val {losses['val']:.4f}")
    xb, yb = get_batch('train')
    _, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.5))
fig.patch.set_facecolor('#F5EEDC'); ax.set_facecolor('#FFF9E9')
ax.plot(xs, train_losses, color='#8B5E3C', lw=2, label='train')
ax.plot(xs, val_losses,   color='#2E4156', lw=2, label='val')
ax.set_title('GPT training on Tiny Shakespeare', color='#2A221B')
ax.set_xlabel('step', color='#7A6B5D'); ax.set_ylabel('loss', color='#7A6B5D')
ax.legend(); ax.grid(alpha=0.3, color='#D4C4B0')
for spine in ax.spines.values(): spine.set_color('#D4C4B0')
plt.tight_layout(); plt.show()

In [ ]:
# Generate a sample
context = torch.zeros((1, 1), dtype=torch.long, device=device)
print(decode(model.generate(context, max_new_tokens=500)[0].tolist()))

## Visualize what the model learned to attend to

Run a single batch through the model, then plot the attention matrix from each head of each layer. Look for patterns:
- Heads that mostly attend to the previous few tokens
- Heads that spike on specific syntactic markers (spaces, newlines, colons)
- Heads that look like diagonals (positional attention)

In [ ]:
model.eval()
prompt = 'ROMEO: Where art thou, Juliet?'
idx = torch.tensor([encode(prompt)], device=device)
with torch.no_grad():
    model(idx)

# show the first 3 heads of each layer
fig, axes = plt.subplots(n_layer, min(3, n_head), figsize=(12, 2.8 * n_layer))
fig.patch.set_facecolor('#F5EEDC')
if n_layer == 1: axes = axes.reshape(1, -1)
for li, block in enumerate(model.blocks):
    for hi, head in enumerate(block.sa.heads[:3]):
        attn = head._last_attn[0].cpu()   # (T, T) from batch 0
        ax = axes[li, hi]
        ax.imshow(attn, cmap='YlOrBr', aspect='auto')
        ax.set_title(f'layer {li} · head {hi}', color='#2A221B', fontsize=11)
        ax.set_xticks(range(len(prompt))); ax.set_yticks(range(len(prompt)))
        ax.set_xticklabels(list(prompt), fontsize=7, color='#2A221B')
        ax.set_yticklabels(list(prompt), fontsize=7, color='#2A221B')
plt.tight_layout(); plt.show()

## You just built a GPT

It's small. It's not good. It hallucinates Shakespearean gibberish. But the *architecture* — tokenization → embeddings → attention blocks → LayerNorm → logits → sampling — is the same architecture as GPT-4, Claude, Llama.

Differences are: bigger model, better data, more training, a few architectural refinements. That's it.

Now the big one: the **[[../10-gpt-addition/index|addition exercise]]**. Teach your transformer to add.